## **Career Recommendation System**

### **Project Information**

**Developer:** Sanusi Shafii

**Matriculation Number:** CSA/2023/27683

**Email:** s.shafii27683@fudutsinma.edu.ng.com

This project implements a Career Recommendation System using machine learning. It helps users discover potential career paths based on their interests and skills, providing personalized recommendations and insights into model performance.

### **Install required libraries**

In [3]:
!pip install ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

# STEP 2: Import all necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully!")
print("=" * 70)

All libraries imported successfully!


### **Load Your Dataset**

In [4]:
import pandas as pd

# Load your dataset (UPDATE THIS WITH YOUR ACTUAL DATASET)
try:
    data_df = pd.read_csv('/content/stud.csv')
    print(f"Dataset loaded successfully!")
    print(f"Dataset shape: {data_df.shape}")
    print("\nDataset columns:")
    print(data_df.columns.tolist())
    print("\nFirst 5 rows:")
    print(data_df.head())

    # --- Start of new logic for language replacement ---
    # Add new language columns, initialized to 0
    data_df['Hausa'] = 0
    data_df['Yoruba'] = 0

    # Transfer interest from 'Hindi' and 'Urdu' to 'Hausa' and 'Yoruba'
    # If a person showed interest in Hindi, set Hausa to 1.
    # If a person showed interest in Urdu, set Yoruba to 1.
    columns_to_drop = []
    if 'Hindi' in data_df.columns:
        data_df.loc[data_df['Hindi'] == 1, 'Hausa'] = 1
        columns_to_drop.append('Hindi')
    if 'Urdu' in data_df.columns:
        data_df.loc[data_df['Urdu'] == 1, 'Yoruba'] = 1
        columns_to_drop.append('Urdu')

    # Drop the original 'Hindi' and 'Urdu' columns
    if columns_to_drop:
        data_df = data_df.drop(columns=columns_to_drop)
        print(f"\nDropped columns: {columns_to_drop}")
        print(f"Added new columns: ['Hausa', 'Yoruba']")
    else:
        print("\n'Hindi' or 'Urdu' columns not found to drop/replace.")

    print("\nUpdated Dataset columns:")
    print(data_df.columns.tolist())
    # --- End of new logic ---

except FileNotFoundError:
    print("ERROR: Dataset file not found!")
    raise

print("=" * 70)

Dataset loaded successfully!
Dataset shape: (3535, 60)

Dataset columns:
['Drawing', 'Dancing', 'Singing', 'Sports', 'Video Game', 'Acting', 'Travelling', 'Gardening', 'Animals', 'Photography', 'Teaching', 'Exercise', 'Coding', 'Electricity Components', 'Mechanic Parts', 'Computer Parts', 'Researching', 'Architecture', 'Historic Collection', 'Botany', 'Zoology', 'Physics', 'Accounting', 'Economics', 'Sociology', 'Geography', 'Psycology', 'History', 'Science', 'Bussiness Education', 'Chemistry', 'Mathematics', 'Biology', 'Makeup', 'Designing', 'Content writing', 'Crafting', 'Literature', 'Reading', 'Cartooning', 'Debating', 'Asrtology', 'Hindi', 'French', 'English', 'Urdu', 'Other Language', 'Solving Puzzles', 'Gymnastics', 'Yoga', 'Engeeniering', 'Doctor', 'Pharmisist', 'Cycling', 'Knitting', 'Director', 'Journalism', 'Bussiness', 'Listening Music', 'Courses']

First 5 rows:
   Drawing  Dancing  Singing  Sports  Video Game  Acting  Travelling  \
0        0        0        0       0    

### **Identify Features and Target Column**

In [5]:
print("\nPlease specify your dataset structure:")
print("-" * 70)

# Get all column names
all_columns = data_df.columns.tolist()
print(f"Total columns in dataset: {len(all_columns)}")

# Assuming 'Courses' is always the target column for career recommendations
target_column = 'Courses'

# Verify if 'Courses' exists in the DataFrame
if target_column not in all_columns:
    print(f"Error: Target column '{target_column}' not found in the dataset.")
    print("Available columns:", all_columns)
    raise ValueError(f"Target column '{target_column}' not found.")

print(f"Target column identified: '{target_column}'")

# Features are all columns except the target column
features = [col for col in all_columns if col != target_column]

print(f"Number of features: {len(features)}")
print(f"Target variable: {target_column}")
print("\nSample features (first 10):")
print(features[:10])

# Check data types (only for features, as target will be encoded later)
print("\nData types check:")
print(f"Features are binary (0/1): {data_df[features].isin([0, 1]).all().all()}")
print(f"Unique values in target: {data_df[target_column].nunique()}")

print("=" * 70)


Please specify your dataset structure:
----------------------------------------------------------------------
Total columns in dataset: 60
Target column identified: 'Courses'
Number of features: 59
Target variable: Courses

Sample features (first 10):
['Drawing', 'Dancing', 'Singing', 'Sports', 'Video Game', 'Acting', 'Travelling', 'Gardening', 'Animals', 'Photography']

Data types check:
Features are binary (0/1): True
Unique values in target: 35


### **Prepare Data for Training**

In [6]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("\nPreparing data for machine learning...")

# Create feature matrix X and target variable Y
X_df = data_df[features]
Y_original = data_df[target_column] # Store original target for display purposes

print(f"Features (X): {X_df.shape}")
print(f"Original Target (Y): {Y_original.shape}")

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the target column
Y = label_encoder.fit_transform(Y_original)

# Store the mapping of encoded labels to original course names
courses_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
# Invert the mapping for easy lookup from encoded label to original name
reverse_courses_mapping = {v: k for k, v in courses_mapping.items()}

print("Target column encoded successfully!")
print("\nEncoded Target (Y):", Y.shape)

# Check for missing values in features
missing_values = X_df.isnull().sum().sum()
if missing_values > 0:
    print(f"Warning: Found {missing_values} missing values in features.")
    # Fill missing values with 0 (assuming binary features)
    X_df = X_df.fillna(0)
    print("Missing values in features filled with 0.")
else:
    print("No missing values found in features.")

print(f"Unique career options (encoded): {len(label_encoder.classes_)}")
print("Career distribution (top 5 encoded labels):")
# Convert Y to a Series for value_counts and then map back for readability
print(pd.Series(Y).map(reverse_courses_mapping).value_counts().head())

print("=" * 70)


Preparing data for machine learning...
Features (X): (3535, 59)
Original Target (Y): (3535,)
Target column encoded successfully!

Encoded Target (Y): (3535,)
No missing values found in features.
Unique career options (encoded): 35
Career distribution (top 5 encoded labels):
BBS- Bachelor of Business Studies                      111
BEM- Bachelor of Event Management                      101
BBA- Bachelor of Business Administration               101
Integrated Law Course- BA + LL.B                       101
BJMC- Bachelor of Journalism and Mass Communication    101
Name: count, dtype: int64


**Train the Machine Learning Model**

In [7]:
print("\nTraining the Career Recommendation Model...")

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_df, Y, test_size=0.2, random_state=42, stratify=Y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Train Random Forest Classifier
model = RandomForestClassifier(
    n_estimators=100, random_state=42, max_depth=10, min_samples_split=5
)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel Training Complete!")
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("=" * 70)

# Display classification report
print("\nDetailed Performance Report:")
# To get a more readable classification report, map back to original labels
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


Training the Career Recommendation Model...
Training samples: 2828
Testing samples: 707

Model Training Complete!
Model Accuracy: 99.72%

Detailed Performance Report:
                                                     precision    recall  f1-score   support

                 Animation, Graphics and Multimedia       1.00      1.00      1.00        20
                   B.Arch- Bachelor of Architecture       1.00      1.00      1.00        20
                        B.Com- Bachelor of Commerce       1.00      0.90      0.95        20
                                              B.Ed.       1.00      1.00      1.00        20
                              B.Sc- Applied Geology       1.00      1.00      1.00        20
                                      B.Sc- Nursing       1.00      1.00      1.00        20
                                    B.Sc. Chemistry       1.00      1.00      1.00        21
                                  B.Sc. Mathematics       1.00      1.00      1.00     

### **Feature Importance Analysis**

In [8]:
# Get feature importance
feature_importance = pd.DataFrame({
    'Feature': X_df.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Most Important Features:")
print(feature_importance.head(15).to_string(index=False))
print("=" * 70)


Top 15 Most Important Features:
       Feature  Importance
  Engeeniering    0.039085
      Teaching    0.035807
       Science    0.034197
        Makeup    0.033861
    Journalism    0.032167
       History    0.030353
        Doctor    0.029807
     Designing    0.029033
     Psycology    0.028462
  Architecture    0.028162
     Chemistry    0.027957
Computer Parts    0.027910
       Physics    0.027590
       English    0.025637
      Exercise    0.024527


### **Interactive Dashboard Class**

In [9]:
class CareerRecommendationDashboard:
    def __init__(self, model, features, label_encoder, feature_importance_df, reverse_courses_mapping):
        self.model = model
        self.features = features
        self.label_encoder = label_encoder # Keep the encoder to decode predictions
        self.courses_list = label_encoder.classes_ # Get original course names
        self.feature_importance_df = feature_importance_df
        self.reverse_courses_mapping = reverse_courses_mapping
        self.checkboxes = {}
        self.output = widgets.Output()

    def create_dashboard(self):
        # Title and description
        title = widgets.HTML(
            value="""
            <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        padding: 30px; border-radius: 10px; margin-bottom: 20px;'>
                <h1 style='text-align: center; color: white; margin: 0;'>
                    Career Recommendation System
                </h1>
                <p style='text-align: center; color: white; font-size: 18px; margin-top: 10px;'>
                    Discover your ideal career path based on your interests and skills
                </p>
            </div>
            """
        )

        subtitle = widgets.HTML(
            value="""
            <div style='background-color: #f0f4f8; padding: 20px; border-radius: 8px; margin-bottom: 20px;'>
                <h3 style='color: #2d3748; margin-top: 0;'>Instructions:</h3>
                <ul style='color: #4a5568; line-height: 1.8;'>
                    <li>Select all the interests and skills that apply to you</li>
                    <li>Click "Get Career Recommendation" to see your personalized results</li>
                    <li>Review your top 5 career matches with confidence scores</li>
                    <li>Use "Show Analytics" to explore data insights and visualizations</li>
                </ul>
            </div>
            """
        )

        categories = self._categorize_features()

        # Create category sections (no accordions) to show all checkboxes by default
        category_sections = []
        for category, items in categories.items():
            category_title = widgets.HTML(value=f"<h4 style='color: #2d3748; margin-top: 15px; margin-bottom: 5px;'>{category}</h4>")
            category_checkboxes = []
            for feature in items:
                if feature in self.features:
                    cb = widgets.Checkbox(
                        value=False,
                        description=feature,
                        style={'description_width': 'initial'},
                        layout=widgets.Layout(width='250px', margin='5px')
                    )
                    self.checkboxes[feature] = cb
                    category_checkboxes.append(cb)

            # Arrange in rows of 3
            rows = []
            for i in range(0, len(category_checkboxes), 3):
                row = widgets.HBox(category_checkboxes[i:i + 3])
                rows.append(row)

            category_sections.append(widgets.VBox([category_title] + rows))

        all_features_box = widgets.VBox(category_sections)

        # Buttons
        predict_btn = widgets.Button(
            description='Get Career Recommendation',
            button_style='success',
            tooltip='Click to get your personalized career recommendations',
            layout=widgets.Layout(width='300px', height='50px', margin='10px')
        )

        clear_btn = widgets.Button(
            description='Clear All',
            button_style='warning',
            tooltip='Clear all selections',
            layout=widgets.Layout(width='200px', height='50px', margin='10px')
        )

        visualize_btn = widgets.Button(
            description='Show Analytics',
            button_style='info',
            tooltip='View detailed analytics and visualizations',
            layout=widgets.Layout(width='250px', height='50px', margin='10px')
        )

        stats_btn = widgets.Button(
            description='Model Statistics',
            button_style='primary',
            tooltip='View model performance statistics',
            layout=widgets.Layout(width='250px', height='50px', margin='10px')
        )

        # Button callbacks
        predict_btn.on_click(self.predict_career)
        clear_btn.on_click(self.clear_selections)
        visualize_btn.on_click(self.show_visualizations)
        stats_btn.on_click(self.show_statistics)

        # Button container
        button_box = widgets.HBox(
            [predict_btn, clear_btn, visualize_btn, stats_btn],
            layout=widgets.Layout(justify_content='center', margin='20px')
        )

        # Display dashboard
        display(title, subtitle, all_features_box, button_box, self.output)

    def _categorize_features(self):
        """Categorize features based on keywords in their names"""
        categories = {
            'General Interests': [],
            'Technical Skills': [],
            'Arts & Creative': [],
            'Sports & Fitness': [],
            'Academic': [],
            'Languages': [],
            'Other': []
        }

        keywords_map = {
            'Technical Skills': ['coding', 'computer', 'programming', 'software', 'tech', 'engineering',
                               'electricity', 'mechanic', 'physics', 'chemistry', 'mathematics'],
            'Arts & Creative': ['drawing', 'dancing', 'singing', 'acting', 'makeup', 'designing',
                              'crafting', 'cartoon', 'director', 'photography', 'art'],
            'Sports & Fitness': ['sports', 'exercise', 'gymnastics', 'yoga', 'cycling', 'fitness'],
            'Academic': ['teaching', 'research', 'science', 'history', 'geography', 'economics',
                        'sociology', 'psychology', 'accounting', 'biology', 'botany', 'zoology'],
            'Languages': ['hindi', 'french', 'english', 'urdu', 'language', 'literature', 'reading',
                         'writing', 'debating', 'hausa', 'yoruba']
        }

        for feature in self.features:
            feature_lower = feature.lower()
            categorized = False

            # Special handling for 'Asrtology' if it exists and needs specific categorization
            if feature == 'Asrtology':
                categories['Academic'].append(feature) # Or another suitable category
                categorized = True
                continue

            for category, keywords in keywords_map.items():
                if any(keyword in feature_lower for keyword in keywords):
                    categories[category].append(feature)
                    categorized = True
                    break

            if not categorized:
                categories['General Interests'].append(feature)

        # Remove empty categories and sort features within categories for consistency
        for category in categories:
            categories[category].sort()
        return {k: v for k, v in categories.items() if v}

    def predict_career(self, b):
        with self.output:
            clear_output()

            # Get user input
            user_input = [1 if self.checkboxes[feature].value else 0 for feature in self.features]
            user_df = pd.DataFrame([user_input], columns=self.features)

            # Check if any interest is selected
            if sum(user_input) == 0:
                display(widgets.HTML(
                    value="""
                    <div style='background-color: #fff3cd; border-left: 5px solid #ffc107;
                                padding: 20px; border-radius: 5px; margin: 20px 0;'>
                        <h3 style='color: #856404; margin-top: 0;'>No Interests Selected</h3>
                        <p style='color: #856404; font-size: 16px;'>
                            Please select at least one interest or skill to get career recommendations.
                        </p>
                    </div>
                    """
                ))
                return

            # Make prediction
            encoded_prediction = self.model.predict(user_df)[0]
            prediction = self.label_encoder.inverse_transform([encoded_prediction])[0]
            probabilities = self.model.predict_proba(user_df)[0]

            # Get top 5 recommendations
            # Get indices of top probabilities
            top_indices = np.argsort(probabilities)[::-1][:5]
            # Decode the top predicted labels and get their probabilities
            top_courses = [
                (self.label_encoder.inverse_transform([idx])[0], probabilities[idx] * 100)
                for idx in top_indices
            ]

            # Create results HTML
            selected = [feat for feat in self.features if self.checkboxes[feat].value]

            results_html = f"""
            <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        padding: 30px; border-radius: 10px; color: white; margin: 20px 0;'>
                <h2 style='text-align: center; margin-top: 0;'>YOUR PERSONALIZED CAREER RECOMMENDATIONS</h2>
            </div>

            <div style='background-color: #e6f7ff; border-left: 5px solid #1890ff;
                        padding: 20px; border-radius: 5px; margin: 20px 0;'>
                <h3 style='color: #0050b3; margin-top: 0;'>PRIMARY RECOMMENDATION</h3>
                <h2 style='color: #0050b3; font-size: 28px;'>{prediction}</h2>
            </div>

            <div style='background-color: white; padding: 25px; border-radius: 10px;
                        box-shadow: 0 2px 8px rgba(0,0,0,0.1); margin: 20px 0;'>
                <h3 style='color: #2d3748;'>Top 5 Career Matches (with confidence scores)</h3>
            """

            for i, (course, prob) in enumerate(top_courses, 1):
                color = '#10b981' if i == 1 else '#3b82f6' if i <= 3 else '#8b5cf6'
                results_html += f"""
                <div style='margin: 15px 0;'>
                    <div style='display: flex; justify-content: space-between; margin-bottom: 5px;'>
                        <span style='font-weight: bold; color: #2d3748;'>{i}. {course}</span>
                        <span style='font-weight: bold; color: {color};'>{prob:.2f}%</span>
                    </div>
                    <div style='background-color: #e5e7eb; height: 25px; border-radius: 12px; overflow: hidden;'>
                        <div style='background: linear-gradient(90deg, {color} 0%, {color}dd 100%);
                                    width: {prob}%; height: 100%; transition: width 0.5s;'></div>
                    </div>
                </div>
                """

            results_html += f"""
            </div>

            <div style='background-color: #f0fdf4; padding: 20px; border-radius: 8px; margin: 20px 0;'>
                <h4 style='color: #166534; margin-top: 0;'>Your Profile Summary</h4>
                <p style='color: #166534; font-size: 16px; margin: 5px 0;'>
                    <strong>Selected Interests:</strong> {len(selected)} out of {len(self.features)}
                </p>
                <p style='color: #166534; font-size: 14px;'>
                    <strong>Your interests:</strong> {', '.join(selected[:15])}
                    {'... and ' + str(len(selected)-15) + ' more' if len(selected) > 15 else ''}
                </p>
            </div>
            """

            display(widgets.HTML(value=results_html))

    def clear_selections(self, b):
        for cb in self.checkboxes.values():
            cb.value = False
        with self.output:
            clear_output()
            display(widgets.HTML(
                value="""
                <div style='background-color: #d1fae5; border-left: 5px solid #10b981;
                            padding: 20px; border-radius: 5px; margin: 20px 0;'>
                    <h3 style='color: #065f46; margin: 0;'>All selections have been cleared!</h3>
                </div>
                """
            ))

    def show_statistics(self, b):
        with self.output:
            clear_output()

            # Create statistics display
            stats_html = f"""
            <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        padding: 30px; border-radius: 10px; color: white; margin: 20px 0;'>
                <h2 style='text-align: center; margin: 0;'>MODEL PERFORMANCE STATISTICS</h2>
            </div>

            <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
                        gap: 20px; margin: 20px 0;'>
                <div style='background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%);
                            padding: 30px; border-radius: 10px; text-align: center; color: white;'>
                    <h3 style='margin: 0; font-size: 18px;'>Model Accuracy</h3>
                    <h1 style='margin: 10px 0; font-size: 48px;'>{accuracy*100:.2f}%</h1>
                </div>

                <div style='background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%);
                            padding: 30px; border-radius: 10px; text-align: center; color: white;'>
                    <h3 style='margin: 0; font-size: 18px;'>Total Features</h3>
                    <h1 style='margin: 10px 0; font-size: 48px;'>{len(self.features)}</h1>
                </div>

                <div style='background: linear-gradient(135deg, #43e97b 0%, #38f9d7 100%);
                            padding: 30px; border-radius: 10px; text-align: center; color: white;'>
                    <h3 style='margin: 0; font-size: 18px;'>Training Samples</h3>
                    <h1 style='margin: 10px 0; font-size: 48px;'>{len(X_train)}</h1>
                </div>

                <div style='background: linear-gradient(135deg, #fa709a 0%, #fee140 100%);
                            padding: 30px; border-radius: 10px; text-align: center; color: white;'>
                    <h3 style='margin: 0; font-size: 18px;'>Career Options</h3>
                    <h1 style='margin: 10px 0; font-size: 48px;'>{len(self.label_encoder.classes_)}</h1>
                </div>
            </div>

            <div style='background-color: white; padding: 25px; border-radius: 10px;
                        box-shadow: 0 2px 8px rgba(0,0,0,0.1); margin: 20px 0;'>
                <h3 style='color: #2d3748;'>Available Career Paths</h3>
                <div style='display: flex; flex-wrap: wrap; gap: 10px; margin-top: 15px;'>
            """

            for course in sorted(self.label_encoder.classes_):
                stats_html += f"""
                    <span style='background-color: #e0e7ff; color: #3730a3; padding: 8px 16px;
                                 border-radius: 20px; font-size: 14px;'>{course}</span>
                """

            stats_html += """
                </div>
            </div>
            """

            display(widgets.HTML(value=stats_html))

    def show_visualizations(self, b):
        with self.output:
            clear_output()

            print("Generating Analytics & Visualizations...\n")

            # Create visualizations
            fig, axes = plt.subplots(2, 2, figsize=(18, 12))
            fig.suptitle('Career Recommendation System - Analytics Dashboard',
                        fontsize=20, fontweight='bold', y=1.00)

            # 1. Feature Importance
            top_15 = self.feature_importance_df.head(15)
            colors = plt.cm.viridis(np.linspace(0, 1, len(top_15)))
            axes[0, 0].barh(top_15['Feature'], top_15['Importance'], color=colors)
            axes[0, 0].set_xlabel('Importance Score', fontsize=12, fontweight='bold')
            axes[0, 0].set_title('Top 15 Most Important Features for Career Prediction',
                                fontsize=14, fontweight='bold', pad=20)
            axes[0, 0].invert_yaxis()
            axes[0, 0].grid(axis='x', alpha=0.3)

            # 2. Career Distribution
            # Use Y_original for distribution as it contains readable course names
            course_counts = Y_original.value_counts().head(10)
            bars = axes[0, 1].bar(range(len(course_counts)), course_counts.values,
                                 color=plt.cm.Set3(np.linspace(0, 1, len(course_counts))))
            axes[0, 1].set_xticks(range(len(course_counts)))
            axes[0, 1].set_xticklabels(course_counts.index, rotation=45, ha='right')
            axes[0, 1].set_ylabel('Number of Students', fontsize=12, fontweight='bold')
            axes[0, 1].set_title('Top 10 Most Popular Career Paths in Dataset',
                                fontsize=14, fontweight='bold', pad=20)
            axes[0, 1].grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bar in bars:
                height = bar.get_height()
                axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                              f'{int(height)}',
                              ha='center', va='bottom', fontweight='bold')

            # 3. User Selection Pattern
            selected_count = sum([1 for cb in self.checkboxes.values() if cb.value])
            categories = ['Selected\nInterests', 'Available\nOptions']
            counts = [selected_count, len(self.checkboxes) - selected_count]
            colors_pie = ['#667eea', '#f0f4f8']
            explode = (0.1, 0)

            wedges, texts, autotexts = axes[1, 0].pie(counts, labels=categories, autopct='%1.1f%%',
                                                      colors=colors_pie, startangle=90, explode=explode,
                                                      shadow=True, textprops={'fontsize': 12, 'fontweight': 'bold'})
            axes[1, 0].set_title('Your Interest Selection Coverage',
                                fontsize=14, fontweight='bold', pad=20)

            # 4. Confusion Matrix (if we have predictions)
            # Map y_test and y_pred back to original labels for confusion matrix for clarity
            y_test_decoded = self.label_encoder.inverse_transform(y_test)
            y_pred_decoded = self.label_encoder.inverse_transform(y_pred)
            cm = confusion_matrix(y_test_decoded, y_pred_decoded, labels=self.label_encoder.classes_)

            # Show top 5 classes for clarity
            # Use the original (decoded) course names to determine top classes
            top_classes_decoded = pd.Series(y_test_decoded).value_counts().head(5).index.tolist()

            # Filter confusion matrix for only these top classes
            # Create a sub-matrix for the top classes
            cm_top_indices = [np.where(self.label_encoder.classes_ == c)[0][0] for c in top_classes_decoded]
            cm_top = cm[cm_top_indices][:, cm_top_indices]

            if len(top_classes_decoded) > 0:
                sns.heatmap(cm_top, annot=True, fmt='d', cmap='Blues',
                           xticklabels=top_classes_decoded, yticklabels=top_classes_decoded,
                           ax=axes[1, 1], cbar_kws={'label': 'Count'})
                axes[1, 1].set_xlabel('Predicted Career', fontsize=12, fontweight='bold')
                axes[1, 1].set_ylabel('Actual Career', fontsize=12, fontweight='bold')
                axes[1, 1].set_title('Confusion Matrix (Top 5 Careers)',
                                    fontsize=14, fontweight='bold', pad=20)
            else:
                axes[1, 1].text(0.5, 0.5, f'Model Accuracy\n\n{accuracy*100:.1f}%',
                               ha='center', va='center', fontsize=40, fontweight='bold',
                               color='#10b981', transform=axes[1, 1].transAxes)
                axes[1, 1].axis('off')
                axes[1, 1].set_title('Overall Model Performance',
                                    fontsize=14, fontweight='bold', pad=20)

            plt.tight_layout()
            plt.show()

            # Print guide
            guide_html = """
            <div style='background-color: #f0f4f8; padding: 25px; border-radius: 10px; margin: 20px 0;'>
                <h3 style='color: #2d3748; margin-top: 0;'>Visualization Guide</h3>
                <ul style='color: #4a5568; line-height: 2; font-size: 15px;'>
                    <li><strong>Top Left:</strong> Shows which interests/skills have the most impact on career predictions</li>
                    <li><strong>Top Right:</strong> Displays the most common career recommendations in the training dataset</li>
                    <li><strong>Bottom Left:</strong> Your current selection coverage (how many interests you've selected)</li>
                    <li><strong>Bottom Right:</strong> Confusion matrix showing model prediction accuracy for top careers</li>
                </ul>
            </div>
            """
            display(widgets.HTML(value=guide_html))

### **Create and Launch the Dashboard**



In [10]:
print("\n" + "=" * 70)
print("LAUNCHING CAREER RECOMMENDATION DASHBOARD")
print("=" * 70)

# Create dashboard instance
dashboard = CareerRecommendationDashboard(model, features, label_encoder, feature_importance, reverse_courses_mapping)

# Display the dashboard
dashboard.create_dashboard()

print("\n" + "=" * 70)
print("Dashboard is ready to use!")
print("=" * 70)
print("\nQUICK START GUIDE:")
print("-" * 70)
print("1. Expand the category accordions to see all interests")
print("2. Check the boxes for interests and skills that match your profile")
print("3. Click 'Get Career Recommendation' for personalized suggestions")
print("4. Click 'Show Analytics' to explore detailed visualizations")
print("5. Click 'Model Statistics' to view performance metrics")
print("6. Click 'Clear All' to reset and try different combinations")
print("=" * 70)
print("\nTIP: Select at least 5-10 interests for better recommendations!")
print("=" * 70)


LAUNCHING CAREER RECOMMENDATION DASHBOARD


HTML(value="\n            <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);\n        …

HTML(value='\n            <div style=\'background-color: #f0f4f8; padding: 20px; border-radius: 8px; margin-bo…

Output()


Dashboard is ready to use!

QUICK START GUIDE:
----------------------------------------------------------------------
1. Expand the category accordions to see all interests
2. Check the boxes for interests and skills that match your profile
3. Click 'Get Career Recommendation' for personalized suggestions
4. Click 'Show Analytics' to explore detailed visualizations
5. Click 'Model Statistics' to view performance metrics
6. Click 'Clear All' to reset and try different combinations

TIP: Select at least 5-10 interests for better recommendations!
